# 数值走查：价值目标与两种策略更新（Python 入门版）

> 配套文件：`examples/update_modes_walkthrough.py`（本 notebook 是它的"教学展开版"）
> 适用：刚入门 Python 的读者；**只用标准库**，不需要 torch/numpy。
> 公式风格：按仓库 `md-latex-rule` 的策略，全篇优先**纯文本公式**（Unicode + 代码段），
> 只有无法用纯文本表达的结构（大分式、矩阵）才会用到 LaTeX——本篇没有这种结构。

**学完本 notebook 你会：**

1. 用 Python 从零实现 `softmax` / `log_softmax` / 交叉熵；
2. 推导两条策略更新公式的梯度，并解释它们为什么不同；
3. 用手工数字跑通：价值目标回溯 → 离线 CE 更新 → 在线 REINFORCE+KL 更新；
4. 用**有限差分**校验自己写的梯度（这是最实用的调试习惯之一）；
5. 理解 batch 平均、r=0 样本、KL 锚"软步长"这些工程细节。

**阅读方式**：从上到下按 `Shift+Enter` 逐格运行；每段代码前都有"逐行拆解"。
遇到不懂的 Python 语法，先看注释和拆解，再看输出——输出本身也是教学材料。

## 目录

| 节 | 内容 | Python 教学点 | 数学 |
| --- | --- | --- | --- |
| §0 | Python 热身 + 表格小工具 | 变量/列表/循环/函数/`enumerate`/`ljust` | — |
| §1 | 价值目标回溯 | `class`/字典/递归/生成器表达式/对象引用/DFS | d(s) 定义、AND 取 min 的推导 |
| §2 | 离线 CE | 函数/列表切片/列表推导 | softmax 雅可比、CE 梯度推导 |
| §3 | 在线 REINFORCE + KL | 默认参数/闭包/就地更新 | 策略梯度定理、平衡点推导 |
| §4 | 有限差分校验 | `lambda`/拷贝陷阱 | 中心差分、误差阶 |
| §5 | batch 平均 | 逐元素运算 | 期望估计、方差、相关样本 |
| §6 | 对照真实仓库代码 | 读真实源码 | 与玩具例子的差异 |
| §7 | 练习清单 | — | 自测 |

> §1 补充小节：**1.3.1** 节点小实验（引用/is/字典增删）、**1.4.1** 生成器+next+min 微型例、**1.4.2** 递归缩进轨迹、**1.5.1** 节点全字段表与 DFS 找路径。


## §0 Python 热身：先认识 6 个最常用的东西

这一节没有数学，先把后面反复用到的 Python 语法过一遍。

```text
1. 变量          z = [0.0, 0.0]        给一个"名字"绑定一个值
2. 列表          [0.0, 0.0]            一串有序的值，用 z[0] 取第一个（下标从 0 开始）
3. 函数          def f(x): ...         把一段逻辑打包，可反复调用
4. 循环          for i in range(3):    重复做 3 次（i = 0,1,2）
5. f-string      f"结果={x:.3f}"       把变量插进字符串；:.3f=保留 3 位小数
6. 断言          assert x > 0          条件不满足就报错——用来固化"数学结论"
```

In [1]:
# 【EX-0.1】热身：变量、列表、索引、f-string
z = [0.0, 0.0]                  # list：两个动作（A、B）的"分数"
name = "A"                      # 字符串
n = 2                           # 整数

print("z =", z)                 # print 可以一次输出多个东西（逗号分隔，会自动加空格）
print(f"name={name}, n={n}")    # f-string：{} 里可以放变量
print(f"第一个数 z[0]={z[0]:.3f}")  # z[0]=第 0 个元素；:.3f=保留 3 位小数

z = [0.0, 0.0]
name=A, n=2
第一个数 z[0]=0.000


### 小工具：把数据打印成表格

数学结论经常是"一组一组的数"，表格最直观。下面这个 `table` 函数后面每节都会用。

```text
输入:  headers = ["动作", "概率"]
       rows    = [["A", "0.5"], ["B", "0.5"]]
输出:  动作 | 概率
      -----+-----
      A    | 0.5
      B    | 0.5
```

实现思路（4 步）：

1. 每列宽度 = max(表头长度, 各单元格长度)——这样才能对齐；
2. `str(x).ljust(w)`：把字符串补空格到宽度 `w`（左对齐）；
3. `" | ".join(列表)`：用 `" | "` 把列表里的字符串连起来；
4. 全部行用 `"\n".join(...)` 拼成一个多行字符串。

In [2]:
# 【EX-0.2】表格小工具：只用最基础的循环和字符串操作
def table(headers, rows):
    # 第 1 步：算每一列的宽度
    widths = []
    for i in range(len(headers)):          # len(headers)=列数；i 依次取 0..列数-1
        w = len(headers[i])                # 先拿表头长度当初始宽度
        for r in rows:                     # 遍历每一行
            if len(str(r[i])) > w:         # str() 把任意类型转成字符串
                w = len(str(r[i]))         # 更新最大宽度
        widths.append(w)                   # 把这一列的宽度记下来

    # 第 2 步：拼表头
    head_cells = []
    for i in range(len(headers)):
        head_cells.append(headers[i].ljust(widths[i]))
    lines = [" | ".join(head_cells)]

    # 第 3 步：分隔线（每个宽度对应一串 "-"）
    lines.append("-+-".join(["-" * w for w in widths]))   # 列表推导：对每个 w 生成 "-"*w

    # 第 4 步：每一行
    for r in rows:
        cells = []
        for i in range(len(r)):
            cells.append(str(r[i]).ljust(widths[i]))
        lines.append(" | ".join(cells))

    return "\n".join(lines)                # 多行字符串，print 时自动换行


print(table(["动作", "概率"], [["A", "0.5"], ["B", "0.5"]]))

动作 | 概率 
---+----
A  | 0.5
B  | 0.5


**动手改一改**：把 `rows` 换成 `[["A", "0.25"], ["B", "0.75"], ["C", "0.0"]]` 再跑一次，
观察第三列宽度是怎么由最长的内容决定的。

## §1 价值目标回溯：让"剩余步数"自下而上传上来

### 1.1 数学：d(s) 是什么

搜索树里的每个节点代表一个证明状态 s。定义：

```text
d(s) = 从状态 s 走到"证明完成"的最小/期望剩余步数   （越接近证明，d 越小）
```

网络不直接回归 d，而是回归 **V(s) = -d(s)**（越大越好）。每个节点的训练标签
`value_target = -d(s)` 靠搜索树**回溯**得到：

```text
终端（已证完）：     value_target = 0
OR 节点：            value_target = -1 + value_target(最优子节点)
AND 节点：           value_target = min(所有子节点的 value_target)
```

**为什么 AND 取 min？** 一条 tactic 把目标拆成 k 个子目标时，父状态要"全部子目标完成"
才算完成；总耗时的下界由**最难（剩余步数最大）**的子目标决定：

```text
d(s_AND) = max_j d(s_j)
因为 value = -d，取负号后 max 变成 min：
value(s_AND) = -d(s_AND) = -max_j d(s_j) = min_j (-d(s_j)) = min_j value(s_j)
```

### 1.2 例子的树（手工数字）

```text
root(OR)
  └─ "split"(tactic) → AND
        ├─ focus0 → x-branch(OR) → "close-x" → x: proved   （1 步）
        └─ focus1 → y-branch(OR) → "progress-y" → y-mid(OR) → "close-y" → y: proved（2 步）
```

手算：

```text
x: proved    value = 0        d = 0
x-branch     value = -1+0 = -1     d = 1
y-mid        value = -1+0 = -1     d = 1
y-branch     value = -1+(-1) = -2  d = 2
AND          value = min(-1,-2) = -2   d = 2   ← 最坏分支决定
root         value = -1+(-2) = -3      d = 3
```

### 1.3 Python：用类（class）表示树节点

`class` 是"自定义一个类型"。这里每个 `Node` 有：

- `name`：显示名；`to_play`："OR" 或 "AND"；`terminal`：是否已证完；
- `children`：**字典** dict，`键=动作字符串 → 值=子节点`；`add()` 往字典里放一项；
- `is_optimal`：是否在最优路径上（回溯时选它）；`value_target`：稍后填的标签。

**注意 `self`**：类的方法第一个参数永远是 `self`，代表"这个对象本身"；
`self.children = {}` 就是"给这个对象挂一个字典属性"。

In [3]:
# 【EX-1.1】树节点：类、初始化方法、字典、方法
class Node:
    def __init__(self, name, to_play="OR", terminal=False):
        self.name = name              # 名字
        self.to_play = to_play        # "OR" 或 "AND"
        self.terminal = terminal      # True=已证完（无子目标）
        self.children = {}            # 字典：动作 -> 子节点
        self.is_optimal = False       # 是否在最优路径上
        self.value_target = None      # 价值标签（稍后计算）

    def add(self, action, child):
        # 加一条边：在这个节点上做 action 会到达 child
        self.children[action] = child


# 快速验证：建两个节点连起来
a = Node("a", terminal=True)
root = Node("root")
root.add("go", a)
print("root 的子节点键:", list(root.children.keys()))   # list(dict) 把键收集成列表
print("子节点名字:", root.children["go"].name)

root 的子节点键: ['go']
子节点名字: a


### 1.3.1 节点小实验：对象是"引用"，字典是"钥匙柜"

三个容易踩的点，直接拿刚才的 `Node` 验证：

1. **对象赋值不是复制**：`b = a` 之后两个名字指向同一个对象；改 `b.name`，`a.name` 跟着变。
2. **`is` 与 `==`**：`is` 问"是不是同一个对象"；普通类没有定义 `==`，默认也按对象身份比较。
3. **字典就是树的结构**：`children` 的键 = 动作，值 = 子节点；`del children[动作]` 就等于"剪掉一条边"。


In [4]:
# 【EX-1.1+】节点小实验：引用、is、字典增删
n1 = Node("n1")
n2 = Node("n1")          # 同名，但是新对象
alias = n1               # 只是别名，不是复制

print("n1 is alias      :", n1 is alias)        # True：同一个对象
print("n1 is n2         :", n1 is n2)           # False：两个不同对象
print("n1 == n2         :", n1 == n2)           # False：默认按身份比较
alias.name = "changed"
print("改 alias.name 后 n1.name :", n1.name)    # changed —— 同一个对象

# 字典：增、查、删
parent = Node("parent")
kid = Node("kid", terminal=True)
parent.add("go", kid)
print("children 键        :", list(parent.children))
print("'go' in children   :", "go" in parent.children)
del parent.children["go"]                         # 剪边 = 从字典删键
print("删除后 children    :", parent.children)


n1 is alias      : True
n1 is n2         : False
n1 == n2         : False
改 alias.name 后 n1.name : changed
children 键        : ['go']
'go' in children   : True
删除后 children    : {}


### 1.4 Python：递归实现回溯

**递归** = 函数自己调用自己。回溯天然递归：要算某个节点的值，先算它子节点的值。

两个新语法：

```text
生成器表达式：  (a, c) for a, c in node.children.items() if c.is_optimal
                items() 同时给 (键, 值)；这是"惰性遍历"，配合 next(...) 取第一个
next(...)：     从生成器里取第一个元素；找不到会抛异常——这里表示"没有最优子节点"
min(...)：      生成器里所有值的最小值
```

### 1.4.1 生成器、`next`、`min` 的五个微型例子

上一节那句 `next((a, c) for a, c in ... if ...)` 把三个语法叠在一起。先拆开单独练：

```text
① [表达式 for x in xs]    列表推导：立刻算完，得到 list
② (表达式 for x in xs)    生成器表达式：惰性求值，而且只能消费一次
③ xs.items()              (键, 值) 成对给出 → 用 a, c 同时接住（元组解包）
④ next(gen)               取第一个；生成器空了会抛 StopIteration
⑤ min(gen)                取最小值；空生成器会直接报错
```


In [5]:
# 【EX-1.2-前置】生成器 / next / min 微型例子
xs = [1, 2, 3]
lst = [x * 2 for x in xs]        # 列表：立刻得到 [2, 4, 6]
gen = (x * 2 for x in xs)        # 生成器：一行都还没算
print("列表:", lst)
print("生成器对象类型:", type(gen).__name__)     # generator
print("next(gen) =", next(gen))                  # 2（取走第一个）
print("next(gen, '空') =", next(gen, "空"))       # 4（取第二个；带默认值不会炸）
print("剩下的转成列表:", list(gen))               # [6]（生成器被消费完了）
print("再 next 给默认值:", next(gen, "空"))       # '空'

d = {"x": 1, "y": 2, "z": 3}
print("items()  :", list(d.items()))            # [('x',1), ('y',2), ('z',3)]
print("values() :", list(d.values()))           # [1, 2, 3]
print("min 值   :", min(d.values()))            # 1
print("min 带 key（按名字长度）:", min(d.keys(), key=len))

# 套到节点上：从 children 里挑第一个 is_optimal 的 (动作, 子节点)
probe = Node("probe")
probe.add("a", Node("a"))
probe.add("b", Node("b"))
probe.children["b"].is_optimal = True
act, child = next((a, c) for a, c in probe.children.items() if c.is_optimal)
print("找到最优子节点:", act, "→", child.name)

try:
    next((a, c) for a, c in probe.children.items() if a == "不存在")
except StopIteration:
    print("空生成器 + next → StopIteration（没有最优子节点时就会这样报错）")


列表: [2, 4, 6]
生成器对象类型: generator
next(gen) = 2
next(gen, '空') = 4
剩下的转成列表: [6]
再 next 给默认值: 空
items()  : [('x', 1), ('y', 2), ('z', 3)]
values() : [1, 2, 3]
min 值   : 1
min 带 key（按名字长度）: x
找到最优子节点: b → b
空生成器 + next → StopIteration（没有最优子节点时就会这样报错）


In [6]:
# 【EX-1.2】递归：官方 compute_value_target 的直译
def compute_value_target(node):
    # 情况 1：终端（证明完成）→ 剩余步数 0
    if node.terminal:
        node.value_target = 0.0
        return 0.0

    # 情况 2：OR 节点 → -1 + 最优子节点
    if node.to_play == "OR":
        # next(生成器)：找到第一个 is_optimal=True 的子节点，返回 (动作, 子节点)
        action, child = next((a, c) for a, c in node.children.items() if c.is_optimal)
        value = -1.0 + compute_value_target(child)       # 递归调用自己

    # 情况 3：AND 节点 → 所有子节点的最小值
    else:
        value = min(compute_value_target(c) for c in node.children.values())

    node.value_target = value        # 写回节点（"记账"）
    return value

### 1.4.2 递归执行轨迹：跟着 7 个节点走一遍

`compute_value_target` 看起来抽象，其实只是**后序 DFS**：先一路递归到叶子，返回时再逐层 `+1` 或取 `min`。
下面给函数装一个"打印器"，用缩进表示递归深度，把每个节点的**进入**和**返回**都打出来：

```text
读法：缩进越深 = 递归越深；同一层里，x 分支先走完，再走 y 分支。
返回时：OR 层 -1（一步代价）；AND 层取两个分支的最小值。
```


In [7]:
# 【EX-1.2+】给递归装打印器：逐层缩进展示调用与返回
def trace_value(node, depth=0):
    pad = "    " * depth
    if node.terminal:
        node.value_target = 0.0
        print(f"{pad}enter {node.name} (terminal) → 返回 0")
        return 0.0
    print(f"{pad}enter {node.name} ({node.to_play})")
    if node.to_play == "OR":
        action, child = next((a, c) for a, c in node.children.items() if c.is_optimal)
        returned = trace_value(child, depth + 1)
        value = -1.0 + returned
        print(f"{pad}返回 {node.name}: -1 + ({returned:.1f}) = {value:.1f}   ← 走 '{action}'")
    else:
        parts = []
        for child in node.children.values():
            parts.append(trace_value(child, depth + 1))
        value = min(parts)
        joined = ", ".join(f"{v:.1f}" for v in parts)
        print(f"{pad}返回 {node.name}: min({joined}) = {value:.1f}")
    node.value_target = value
    return value

# 重新建一棵一模一样的新树（避免复用已经算过 value_target 的旧节点）
lx = Node("x: proved", terminal=True); nx = Node("x-branch(OR)"); nx.add("close-x", lx)
ly = Node("y: proved", terminal=True); ny2 = Node("y-mid(OR)"); ny2.add("close-y", ly)
ny = Node("y-branch(OR)"); ny.add("progress-y", ny2)
an = Node("split(AND)", to_play="AND"); an.add("focus 0", nx); an.add("focus 1", ny)
rt = Node("root(OR)"); rt.add("split", an)
for node in (rt, an, nx, ny, ny2, lx, ly):
    node.is_optimal = True

value = trace_value(rt)
print()
print(f"最终 root.value_target = {value:.1f} → d = {-value:.1f} 步")
assert value == -3.0


enter root(OR) (OR)
    enter split(AND) (AND)
        enter x-branch(OR) (OR)
            enter x: proved (terminal) → 返回 0
        返回 x-branch(OR): -1 + (0.0) = -1.0   ← 走 'close-x'
        enter y-branch(OR) (OR)
            enter y-mid(OR) (OR)
                enter y: proved (terminal) → 返回 0
            返回 y-mid(OR): -1 + (0.0) = -1.0   ← 走 'close-y'
        返回 y-branch(OR): -1 + (-1.0) = -2.0   ← 走 'progress-y'
    返回 split(AND): min(-1.0, -2.0) = -2.0
返回 root(OR): -1 + (-2.0) = -3.0   ← 走 'split'

最终 root.value_target = -3.0 → d = 3.0 步


### 1.5 组装树、标记最优路径、计算并打印

建树就是上一节的 `Node(...)` + `add(...)` 的重复；
"标记最优路径"是把 `is_optimal=True` 打在决定价值的那条路径上（模拟搜索找到解之后的状态）。

In [8]:
# 【EX-1.3】建树（就是 §1.2 的那棵树）
leaf_x = Node("x: proved", terminal=True)
node_x = Node("x-branch(OR)")
node_x.add("close-x", leaf_x)

leaf_y = Node("y: proved", terminal=True)
node_y2 = Node("y-mid(OR)")
node_y2.add("close-y", leaf_y)
node_y = Node("y-branch(OR)")
node_y.add("progress-y", node_y2)

and_node = Node("split(AND)", to_play="AND")
and_node.add("focus 0", node_x)
and_node.add("focus 1", node_y)

root = Node("root(OR)")
root.add("split", and_node)

# 标记最优路径（人在模拟"搜索已经找到解"）
for node in (root, and_node, node_x, node_y, node_y2, leaf_x, leaf_y):
    node.is_optimal = True

value = compute_value_target(root)
print(f"root.value_target = {value:.1f}   →   d(root) = {-value:.1f} 步")
assert value == -3.0, "回溯结果应为 -3"

root.value_target = -3.0   →   d(root) = 3.0 步


In [9]:
# 【EX-1.4】把每个节点的结果打成表
rows = []
for name, node in (("x: proved", leaf_x), ("y: proved", leaf_y), ("y-mid(OR)", node_y2),
                   ("y-branch(OR)", node_y), ("x-branch(OR)", node_x),
                   ("split(AND)", and_node), ("root(OR)", root)):
    d = abs(float(node.value_target))          # value_target ≤ 0，d = -value
    kind = "terminal" if node.terminal else node.to_play
    rows.append([name, f"d = {d:.1f}", kind])

print(table(["节点", "剩余步数", "类型"], rows))
print()
print("观察：AND 的 d=2 = max(1,2)，把整棵树的 d 从 1 顶到了 3——最坏分支决定总体。")

节点           | 剩余步数    | 类型      
-------------+---------+---------
x: proved    | d = 0.0 | terminal
y: proved    | d = 0.0 | terminal
y-mid(OR)    | d = 1.0 | OR      
y-branch(OR) | d = 2.0 | OR      
x-branch(OR) | d = 1.0 | OR      
split(AND)   | d = 2.0 | AND     
root(OR)     | d = 3.0 | OR      

观察：AND 的 d=2 = max(1,2)，把整棵树的 d 从 1 顶到了 3——最坏分支决定总体。


### 1.5.1 逐个节点拆解：每个 `Node` 到底存了什么

用一张表把 7 个节点的全部字段打出来，树的样子是：

```text
root(OR) d=3
└─ "split" ── split(AND) d=2
   ├─ "focus 0" ── x-branch(OR) d=1 ── "close-x" ── x: proved d=0
   └─ "focus 1" ── y-branch(OR) d=2 ── "progress-y" ── y-mid(OR) d=1 ── "close-y" ── y: proved d=0
```

再顺手练一次递归：给一个 `find_path`，从 root 用 DFS 找"到某个节点的动作序列"——
这正好演示"递归 + 累积参数（trail）"的写法。


In [10]:
# 【EX-1.4+】节点全字段表 + 用 DFS 找根到叶的动作路径
def find_path(node, target_name, trail=()):
    """返回 root→target 的动作元组；找不到返回 None。"""
    if node.name == target_name:
        return list(trail)
    for action, child in node.children.items():
        found = find_path(child, target_name, trail + (action,))
        if found is not None:
            return found
    return None

rows = []
for name, node in (("root(OR)", root), ("split(AND)", and_node), ("x-branch(OR)", node_x),
                   ("y-branch(OR)", node_y), ("y-mid(OR)", node_y2),
                   ("x: proved", leaf_x), ("y: proved", leaf_y)):
    rows.append([
        name,
        node.to_play if not node.terminal else "—",
        "是" if node.terminal else "否",
        str(list(node.children.keys())),
        "是" if node.is_optimal else "否",
        f"{node.value_target:.1f}" if node.value_target is not None else "—",
    ])
print(table(["节点", "to_play", "terminal", "children 键", "最优", "value_target"], rows))
print()
print("root → y: proved 路径:", find_path(root, "y: proved"))
print("root → x: proved 路径:", find_path(root, "x: proved"))


节点           | to_play | terminal | children 键             | 最优 | value_target
-------------+---------+----------+------------------------+----+-------------
root(OR)     | OR      | 否        | ['split']              | 是  | -3.0        
split(AND)   | AND     | 否        | ['focus 0', 'focus 1'] | 是  | -2.0        
x-branch(OR) | OR      | 否        | ['close-x']            | 是  | -1.0        
y-branch(OR) | OR      | 否        | ['progress-y']         | 是  | -2.0        
y-mid(OR)    | OR      | 否        | ['close-y']            | 是  | -1.0        
x: proved    | —       | 是        | []                     | 是  | 0.0         
y: proved    | —       | 是        | []                     | 是  | 0.0         

root → y: proved 路径: ['split', 'focus 1', 'progress-y', 'close-y']
root → x: proved 路径: ['split', 'focus 0', 'close-x']


**§1 小结**

- 数学：`value_target` 回溯 = 终端 0 / OR `-1+子` / AND `min`；AND 取 min 来自 `d_AND = max_j d_j`；
- Python：`class`（`__init__`、`self`、方法）、`dict`、`recursion`、生成器表达式 `next(...)`、`min(...)`；
- 习惯：算完用 `assert` 把结论钉住（之后改代码时立刻能发现回归）。

## §2 离线 CE：模仿"搜索选中的动作"

### 2.1 数学：softmax、log-softmax

网络对 k 个候选动作吐出 k 个 logits `z = [z₁, …, z_k]`，softmax 把它变成概率：

```text
pᵢ = exp(zᵢ) / Σⱼ exp(zⱼ)          （第 i 个动作被选中的概率）
log pᵢ = zᵢ − logsumexp(z)，其中 logsumexp(z) = log Σⱼ exp(zⱼ)
```

**数值稳定技巧**：`exp` 很大时会溢出。由于 softmax 对"全体减同一个常数"不变，
实现时先减最大值 `m = max(z)`：

```text
pᵢ = exp(zᵢ−m) / Σⱼ exp(zⱼ−m)      （结果完全一样，但不会溢出）
```

### 2.2 数学：交叉熵（CE）与梯度推导

搜索选中了动作 `t`（teacher）。监督目标就是"把 t 的概率推高"，损失：

```text
L = −log p_t
```

**第一步**：先求 softmax 的偏导。记 `S = Σⱼ exp(zⱼ)`，`pᵢ = exp(zᵢ)/S`：

```text
∂pᵢ/∂zⱼ = (δᵢⱼ·exp(zᵢ)·S − exp(zᵢ)·exp(zⱼ)) / S²
        = pᵢ·δᵢⱼ − pᵢ·pⱼ
        = pᵢ(δᵢⱼ − pⱼ)                    （δᵢⱼ：i=j 时 1，否则 0）
```

**第二步**：代入 L = −log p_t：

```text
∂L/∂zⱼ = −(1/p_t)·∂p_t/∂zⱼ
        = −(1/p_t)·p_t·(δ_tⱼ − pⱼ)
        = pⱼ − δ_tⱼ

即：  ∂L/∂z = p − e_t                （e_t 是 one-hot：t 位置 1，其余 0）
```

**直觉**：对 teacher 动作 t，梯度 `p_t−1 < 0`，梯度下降 `z ← z − η·grad` 会把 `z_t` **推高**；
其余动作 `p_j > 0`，会把它们的 `z_j` **压低**。概率之和恒为 1，此消彼长。

### 2.3 例子的数字

两个动作 A、B，初始 `z = [0, 0]`，teacher = B，学习率 η = 2.0，跑 8 步。

In [11]:
# 【EX-2.1】softmax 与 log_softmax：逐行拆解
import math

def softmax(z):
    m = max(z)                                     # 1) 找最大值（数值稳定）
    exps = []
    for zi in z:                                   # 2) 每个元素求 exp(zᵢ−m)
        exps.append(math.exp(zi - m))
    s = sum(exps)                                  # 3) 指数和
    probs = []
    for e in exps:                                 # 4) 每个指数除以和
        probs.append(e / s)
    return probs


def log_softmax(z):
    m = max(z)
    # sum(... for ...) 是"生成器表达式"：懒计算所有 exp，再求和
    lse = m + math.log(sum(math.exp(zi - m) for zi in z))   # logsumexp
    return [zi - lse for zi in z]                  # 列表推导：每个 log 概率


print("softmax([0,0])      =", softmax([0.0, 0.0]))
print("log_softmax([0,0])  =", [round(v, 4) for v in log_softmax([0.0, 0.0])])
print("两种写法一致性检查（差 < 1e-12）:",
      max(abs(math.log(p) - lp) for p, lp in zip(softmax([0.3, -0.2]), log_softmax([0.3, -0.2]))))

softmax([0,0])      = [0.5, 0.5]
log_softmax([0,0])  = [-0.6931, -0.6931]
两种写法一致性检查（差 < 1e-12）: 1.1102230246251565e-16


**拆解要点**

- `max(z)` / `sum(x)` 是内置函数；`math.exp` / `math.log` 来自 `math` 模块；
- 列表推导 `[zi - lse for zi in z]` 等价于"建一个空列表 + 循环 append"，但更短；
- `zip(a, b)` 把两个列表按位置配对，常用来逐元素比较；
- 最后一行验证了"`log(softmax(z))` 等于 `log_softmax(z)`"（数学恒等式的实现级校验）。

In [12]:
# 【EX-2.2】CE 损失与解析梯度：公式 ∂L/∂z = p − e_t 的直译
def ce_loss(z, teacher):
    return -log_softmax(z)[teacher]        # L = −log p_teacher

def ce_grad(z, teacher):
    p = softmax(z)                          # p
    g = p[:]                                # 切片复制：g 是新列表，不会改到 p
    g[teacher] -= 1.0                       # g = p − e_teacher（只改 teacher 那一维）
    return g


z = [0.0, 0.0]
print("L =", ce_loss(z, teacher=1))
print("grad =", ce_grad(z, teacher=1), "  ← [p_A−0, p_B−1] = [+0.5, −0.5]")

L = 0.6931471805599453
grad = [0.5, -0.5]   ← [p_A−0, p_B−1] = [+0.5, −0.5]


In [13]:
# 【EX-2.3】CE 训练循环：8 步，每步打印 p / L / 梯度
z = [0.0, 0.0]
teacher = 1          # 0=A, 1=B
eta = 2.0            # 学习率（故意取大，8 步就能看到明显变化）
rows = []

for step in range(8):
    p = softmax(z)
    loss = ce_loss(z, teacher)
    grad = ce_grad(z, teacher)
    rows.append([str(step), f"[{p[0]:.4f}, {p[1]:.4f}]", f"{loss:.4f}",
                 f"[{grad[0]:+.4f}, {grad[1]:+.4f}]"])
    # 梯度下降一步：z ← z − η·grad
    for i in range(len(z)):
        z[i] = z[i] - eta * grad[i]

print(table(["step", "p = [A, B]", "L", "grad"], rows))
print()
print(f"8 步后 p(B) = {softmax(z)[1]:.4f}，L = {ce_loss(z, teacher):.4f}")
assert softmax(z)[1] > 0.95

step | p = [A, B]       | L      | grad              
-----+------------------+--------+-------------------
0    | [0.5000, 0.5000] | 0.6931 | [+0.5000, -0.5000]
1    | [0.1192, 0.8808] | 0.1269 | [+0.1192, -0.1192]
2    | [0.0775, 0.9225] | 0.0807 | [+0.0775, -0.0775]
3    | [0.0580, 0.9420] | 0.0598 | [+0.0580, -0.0580]
4    | [0.0466, 0.9534] | 0.0477 | [+0.0466, -0.0466]
5    | [0.0390, 0.9610] | 0.0397 | [+0.0390, -0.0390]
6    | [0.0335, 0.9665] | 0.0341 | [+0.0335, -0.0335]
7    | [0.0294, 0.9706] | 0.0299 | [+0.0294, -0.0294]

8 步后 p(B) = 0.9737，L = 0.0266


**§2 小结**

- 数学：softmax 雅可比 `∂pᵢ/∂zⱼ = pᵢ(δᵢⱼ−pⱼ)` → CE 梯度 `∂L/∂z = p − e_t`；
- Python：函数、默认参数、切片复制 `p[:]`、列表推导、`zip`、`range`、就地修改列表；
- 现象：CE 的梯度方向**恒定**指向 teacher（与奖励无关）——这就是"监督/模仿"的含义。

对照：真实仓库 `update_offline/learner.py` 的 `_policy_ce` 就是对 tactic 的**每个 token**
做同样的 CE（prompt 段用 `-100` 掩码），公式完全一致，只是那里是序列版本。

## §3 在线更新：REINFORCE + KL 锚

### 3.1 数学：策略梯度定理（一句话版）

现在没有 teacher，只有验证器给的一个标量奖励 `r`。目标：

```text
J(θ) = E_{a ~ πθ}[ r(a) ]              让"期望回报"变大
```

策略梯度定理：`∇J(θ) = E[ r · ∇log πθ(a|s) ]`。
直觉：**如果这次动作拿到了正奖励，就提高它的对数概率；负奖励就降低**。
实际中只有一条采样 `(a, r)`，就用 `r · ∇log πθ(a|s)` 估计梯度（REINFORCE）。

### 3.2 数学：本仓库的具体形式（Δ 与 KL 锚）

```text
Δ = log πθ(a|s) − log π_old(a|s)        （a 是采样到的动作；log π_old 是采样时的值）
L = −r·Δ + β·Δ²                          （β 是 KL 锚系数，默认 0.05）
```

对 logit 参数 z 求导（用 §2 的结论 `∂logπ(a)/∂z = e_a − p`）：

```text
∂L/∂z = −r·(e_a − p) + β·2Δ·(e_a − p)
      = (−r + 2βΔ)·(e_a − p)
      = (−r + 2βΔ)·∇log πθ(a|s)

梯度下降更新：  θ ← θ − η·∇L = θ + η·(r − 2βΔ)·∇log πθ(a|s)
```

**读数**：

- `r` 决定方向：r=+1 推高这条动作，r=−1 压低；
- `2βΔ` 是"锚"带来的自适应修正：新策略比旧策略更看好这条动作（Δ>0）时，修正项把它往回拉；
- 当 `−r + 2βΔ = 0`，即 **Δ* = r / (2β)** 时，这条动作不再被推动（局部平衡）。

**平衡点的概率形式**：

```text
Δ* = r/(2β)  ⇒  p*(a) = p_old(a)·e^{Δ*} = p_old(a)·e^{r/(2β)}
```

若 `e^{r/(2β)}` 太大（比如 r=1、β=0.05 时 e²⁰），平衡点远超概率上限 1 → **锚实际上不设限**；
β 取大一些才会看到明显的"封顶"效果。下面两种 β 各跑 400 步验证。

In [14]:
# 【EX-3.1】在线损失与解析梯度：公式的直译
def online_loss(z, action, logp_old, r, beta):
    delta = log_softmax(z)[action] - logp_old      # Δ
    return -r * delta + beta * delta * delta       # L = −rΔ + βΔ²


def online_grad(z, action, logp_old, r, beta):
    delta = log_softmax(z)[action] - logp_old
    p = softmax(z)
    coef = -r + 2.0 * beta * delta                 # 有效系数
    e = [0.0] * len(z)                             # e_a：one-hot，长度随动作数自动变
    e[action] = 1.0
    g = []
    for i in range(len(z)):
        g.append(coef * (e[i] - p[i]))             # coef·(e_a − p)
    return g

In [15]:
# 【EX-3.2】跑 N 步在线更新，记录每一步的 (p, Δ, 系数)
def run_online(z0, action, r, beta, eta=0.05, steps=400):
    z = list(z0)                                   # 复制：不修改调用方传进来的列表
    logp_old = log_softmax(z0)[action]             # 采样时刻的旧对数概率（常数）
    history = []
    for _ in range(steps):                         # "_" 表示循环变量用不到
        p = softmax(z)
        delta = log_softmax(z)[action] - logp_old
        coef = -r + 2.0 * beta * delta
        history.append([p, delta, coef])
        grad = online_grad(z, action, logp_old, r, beta)
        for i in range(len(z)):
            z[i] = z[i] - eta * grad[i]            # 梯度下降
    return z, history


# 两个 β 对照：小 β（锚≈不设限） vs 大 β（锚明显）
for beta in (0.05, 5.0):
    z_final, hist = run_online([0.0, 0.0], action=0, r=1.0, beta=beta)
    p_final = softmax(z_final)[0]
    delta_star = 1.0 / (2.0 * beta)
    p_star = 0.5 * math.exp(delta_star)
    note = "（超出概率范围 → 不设限）" if p_star > 1.0 else ""
    print(f"β={beta:<5} 400 步后 p(A)={p_final:.4f}；闭式平衡 p*(A)={p_star:.4f}{note}")
    rows = []
    for i in (0, 5, 50, 399):
        p, delta, coef = hist[i]
        rows.append([str(i), f"{p[0]:.4f}", f"{delta:+.4f}", f"{coef:+.4f}"])
    print(table(["step", "p(A)", "Δ", "系数 −r+2βΔ"], rows))
    print()

β=0.05  400 步后 p(A)=0.9724；闭式平衡 p*(A)=11013.2329（超出概率范围 → 不设限）
step | p(A)   | Δ       | 系数 −r+2βΔ
-----+--------+---------+----------
0    | 0.5000 | +0.0000 | -1.0000  
5    | 0.5589 | +0.1114 | -0.9889  
50   | 0.8150 | +0.4886 | -0.9511  
399  | 0.9723 | +0.6651 | -0.9335  

β=5.0   400 步后 p(A)=0.5526；闭式平衡 p*(A)=0.5526
step | p(A)   | Δ       | 系数 −r+2βΔ
-----+--------+---------+----------
0    | 0.5000 | +0.0000 | -1.0000  
5    | 0.5378 | +0.0728 | -0.2716  
50   | 0.5526 | +0.1000 | -0.0000  
399  | 0.5526 | +0.1000 | -0.0000  



In [16]:
# 【EX-3.3】大 β 时用闭式平衡做断言：p*(A) = 0.5·e^(1/(2β))
z_final, _ = run_online([0.0, 0.0], action=0, r=1.0, beta=5.0)
p_final = softmax(z_final)[0]
expected = 0.5 * math.exp(0.1)
print(f"模拟值 {p_final:.6f}  vs  闭式值 {expected:.6f}  差 {abs(p_final-expected):.2e}")
assert abs(p_final - expected) < 0.01

模拟值 0.552585  vs  闭式值 0.552585  差 0.00e+00


**§3 小结**

- 数学：策略梯度定理 → REINFORCE；KL 锚给出"软步长"，平衡点 `Δ* = r/(2β)`；
- Python：函数默认参数（`eta=0.05, steps=400`）、`list(z0)` 复制、`for _ in range(...)`、
  就地更新列表元素；
- 与 CE 的本质差别：这里**没有 teacher**，梯度方向由标量奖励 r 决定，
  而且每条轨迹只有一个 r（"稀疏、高方差"的来源，§5 会看到 batch 如何缓解）。

## §4 有限差分：给梯度做"体检"

### 4.1 数学

导数定义 `f'(x) = lim_{ε→0} [f(x+ε) − f(x−ε)] / (2ε)`。取很小的 ε（如 1e-6）：

```text
中心差分：  f'(x) ≈ [f(x+ε) − f(x−ε)] / (2ε)     截断误差 O(ε²)
```

它的意义：**不需要推导**就能得到梯度。把"手推的解析梯度"和"数值梯度"逐分量比对，
误差 < 1e-6 就说明我们的公式/实现没写错。

### 4.2 Python：`lambda`（匿名函数）

`numeric_grad` 需要传入一个"关于 z 的函数"。但我们已有的 `ce_loss(z, teacher)` 有两个参数，
于是用 `lambda` 现场包一个只接收 z 的函数：

```text
lambda zz: ce_loss(zz, 1)          # 等价于 def f(zz): return ce_loss(zz, 1)
```

**陷阱**：在 `numeric_grad` 里扰动 `z[i]` 时必须先 `list(z)` 复制，
否则会改到原列表（Python 的列表是"引用传递"，函数内外指向同一个对象）。

In [17]:
# 【EX-4.1】中心差分数值梯度
def numeric_grad(f, z, eps=1e-6):
    out = []
    for i in range(len(z)):
        zp = list(z)            # 复制一份再扰动（重要！）
        zp[i] += eps
        zm = list(z)            # 再复制一份
        zm[i] -= eps
        out.append((f(zp) - f(zm)) / (2 * eps))
    return out


def max_abs_diff(a, b):
    return max(abs(x - y) for x, y in zip(a, b))


z = [0.3, -0.2]

# CE：解析 vs 数值
ana = ce_grad(z, teacher=1)
num = numeric_grad(lambda zz: ce_loss(zz, 1), z)
print("CE   解析:", [f"{v:+.6f}" for v in ana])
print("CE   数值:", [f"{v:+.6f}" for v in num])
assert max_abs_diff(ana, num) < 1e-6

# 在线：解析 vs 数值（logp_old 用 0.5 的对数）
ana2 = online_grad(z, action=0, logp_old=math.log(0.5), r=1.0, beta=0.05)
num2 = numeric_grad(lambda zz: online_loss(zz, 0, math.log(0.5), 1.0, 0.05), z)
print("在线 解析:", [f"{v:+.6f}" for v in ana2])
print("在线 数值:", [f"{v:+.6f}" for v in num2])
assert max_abs_diff(ana2, num2) < 1e-6

print("\n两条梯度都通过 1e-6 校验 ✓")

CE   解析: ['+0.622459', '-0.622459']
CE   数值: ['+0.622459', '-0.622459']
在线 解析: ['-0.369270', '+0.369270']
在线 数值: ['-0.369270', '+0.369270']

两条梯度都通过 1e-6 校验 ✓


**§4 小结**：解析梯度（公式推导 + 手写实现）永远可能有符号/系数错误；
有限差分是"零推导的正确性参照"。工程里每次改 loss，先跑一遍这个检查。

## §5 batch 平均：为什么 batch size 很重要

### 5.1 数学

单条样本的梯度 `gᵢ` 是期望梯度的一个**估计**。若把 N 条样本求平均：

```text
g_batch = (1/N)·Σᵢ gᵢ
```

- 独立同分布时：`Var(g_batch) ≈ Var(gᵢ)/N`（方差随 N 下降）；
- **但同一条轨迹里的多个事件共享同一个 r**，高度相关——有效样本数更接近"轨迹条数"，
  而不是"事件条数"。这就是"按问题数凑 batch 比按事件数更有效"的原因。

### 5.2 r=0 的样本不一定没梯度

```text
r=0：policy 项 −rΔ 为 0；
但若 Δ≠0（策略已偏离采样时的策略），KL 项 2βΔ 仍在：
  ∂L/∂z = 2βΔ·(e_a − p)     ← 只把 log-prob 拉回旧策略
```

### 5.3 两个动作的小陷阱

只有 2 个动作时 `p_A + p_B = 1`，概率空间只剩 1 维自由度：
"推高 A"与"压低 B"是**同一个方向**的梯度。动作 ≥3 个时正/负奖励才会出现不同分量。

In [18]:
# 【EX-5.1】三条样本的梯度与 batch 平均
z = [0.0, 0.0]                 # 恰好等于采样时的 z（Δ=0）
logp_old = math.log(0.5)
samples = [(0, +1.0, "A 成功"), (1, -1.0, "B 失败"), (0, 0.0, "A 未闭合 r=0")]

grads = []
rows = []
for action, r, tag in samples:
    g = online_grad(z, action, logp_old, r, beta=0.05)
    grads.append(g)
    rows.append([tag, f"r={r:+.1f}", f"[{g[0]:+.4f}, {g[1]:+.4f}]"])

mean = [sum(g[i] for g in grads) / len(grads) for i in range(2)]     # 逐分量平均
rows.append(["batch 平均", "-", f"[{mean[0]:+.4f}, {mean[1]:+.4f}]"])
print(table(["样本", "奖励", "policy 梯度 coef·(e−p)"], rows))
print()
print("提示：2 个动作时 [−0.5, +0.5] 表示 z_A 降、z_B 升；换 3 个动作就能看到正负样本的分量差异。")

样本        | 奖励     | policy 梯度 coef·(e−p)
----------+--------+---------------------
A 成功      | r=+1.0 | [-0.5000, +0.5000]  
B 失败      | r=-1.0 | [-0.5000, +0.5000]  
A 未闭合 r=0 | r=+0.0 | [+0.0000, -0.0000]  
batch 平均  | -      | [-0.3333, +0.3333]  

提示：2 个动作时 [−0.5, +0.5] 表示 z_A 降、z_B 升；换 3 个动作就能看到正负样本的分量差异。


In [19]:
# 【EX-5.2】Δ≠0 时，r=0 的样本只剩 KL 项梯度（解析式：2βΔ·(e−p)）
logp_old_shifted = logp_old + 0.2       # 假装当前策略比采样时更看好 A
g0 = online_grad(z, action=0, logp_old=logp_old_shifted, r=0.0, beta=0.05)

delta = log_softmax(z)[0] - logp_old_shifted
expected = [2 * 0.05 * delta * (1.0 - 0.5),
            2 * 0.05 * delta * (0.0 - 0.5)]
print(f"r=0, Δ={delta:+.4f}")
print("梯度:", [f"{v:+.6f}" for v in g0])
print("解析:", [f"{v:+.6f}" for v in expected])
assert max_abs_diff(g0, expected) < 1e-12
print("\n没有奖励信号时，KL 锚仍在把 log-prob 拉回旧策略 ✓")

r=0, Δ=-0.2000
梯度: ['-0.010000', '+0.010000']
解析: ['-0.010000', '+0.010000']

没有奖励信号时，KL 锚仍在把 log-prob 拉回旧策略 ✓


**§5 小结**：batch 平均是策略梯度降方差的基本手段；同轨迹样本相关 → 按"问题数"计数更实；
r=0 ≠ 无梯度（锚项还在）。这直接解释了 `update_online/learner.py` 里
`learn_batch_size` / `micro_batch_size` 两个参数的意义。

## §6 对照真实仓库代码：玩具例子和真实实现差在哪

| 玩具例子（本 notebook） | 真实实现 | 差异 |
| --- | --- | --- |
| 动作 = 一个下标（2~3 个候选） | 动作 = 一整条 tactic 的 token 序列 | CE 对每个 token 算，online 的 logp 是 token 对数概率之**和** |
| 手写 `softmax([z1,z2])` | 7B 网络 + LM head 输出 `[B,T,V]` logits | 数学相同，规模不同 |
| CE：`-log_softmax(z)[t]` | `update_offline/learner.py::_policy_ce`：shift + `ignore_index=-100` 掩码 | prompt 段不产生损失 |
| online：`-rΔ+βΔ²` | `update_online/learner.py::update`：同样的公式 + 快照/回滚 | 工程安全机制 |
| 价值目标：手工树 | `alphaproof/targets/value_targets.py::compute_value_target` | 同一套 0 / -1+子 / min 语义 |

**真实代码关键行速览（文字版）**：

```text
# update_offline/learner.py
shift_logits = logits[:, :-1, :]; shift_labels = labels[:, 1:]
loss = CE(shift_logits, shift_labels, ignore_index=-100)   # 逐 token CE
total = policy_mean + value_coef * value_mean              # value_coef = 1e-3

# update_online/learner.py
logp = -token_nll_sum(logits, labels)          # 序列对数概率 = −Σ NLL
delta = logp - logprob_old
policy_total += -reward * delta + beta_kl * delta**2
total = policy_mean + value_coef * value_mean
```

配置文件：`alphaproof/config.py`（`beta_kl=0.05`、`value_coef=1e-3`、`learn_batch_size` 等）。

## §7 练习清单（改一个数、重跑、解释现象）

1. **换 teacher**：§2 里把 `teacher=1` 改成 `0`，重跑，p(A) 会怎么变？
2. **换奖励**：§3 里把 `r=1.0` 改成 `-1.0`，观察 p(A) 是升还是降，并用有效系数解释。
3. **第三个动作**：把 `z` 改成 `[0,0,0]`，重跑 §3；再把三条样本里加一条 `(2, +1.0, "C 成功")`，
   观察正/负奖励的分量不再重合。
4. **找平衡点**：β 从 0.05 调到 2.0，先手算 `p* = 0.5·e^{1/(2β)} = 0.5·e^{0.25}`，
   再用模拟验证（提示：把 `steps` 调到 1000）。
5. **加一层树**：给 §1 的 root 之上再加一个 OR 节点，验证 `value = -1 + (-3) = -4`。
6. **梯度体检**：改掉 `online_grad` 里的 `coef` 符号（故意写错），跑 §4 的有限差分，
   看断言如何立刻抓住这个 bug，然后改回来。

**自测题**

- 为什么 AND 节点取 min 而不是 max？（从 `d_AND = max_j d_j` 推）
- CE 的梯度方向和 r 有关系吗？online 呢？
- β 的作用是什么？β=0 时在线更新会退化成什么？
- `learn_batch_size=1` 与 `=50` 的梯度方差大约差多少倍？（按独立样本粗算）

## 结语

- 源文件：`examples/update_modes_walkthrough.py`（相同逻辑的脚本版）；
- 本 notebook 可直接重新执行（纯标准库，不依赖 torch）；
- 相关仓库组件：`update_offline/`、`update_online/`、`alphaproof/mcts/`、`alphaproof/targets/`；
- 遇到公式表述：优先纯文本（本 notebook 的写法），大分式/矩阵才用 LaTeX。